# ✅ Python Data Quality & Observability — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A data quality system is a quality inspector on an assembly line. Schema validation is the incoming inspection gate — if the bolt is the wrong size, it never enters production. Statistical profiling is the caliper that measures dimensions and flags drift. Anomaly detection is the pressure sensor — it watches the flow rate and screams when something drops 40%. Lineage is the paper trail — when defective parts reach the customer, you trace back to which supplier, which batch, which day.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Data Quality? The Visual Model](#1) |
| 2 | [Core Concepts — Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Schema Validation](#5) |
| 6 | [Pattern 2: Statistical Profiling](#6) |
| 7 | [Pattern 3: Expectation-Based Validation (Great Expectations style)](#7) |
| 8 | [Pattern 4: Anomaly Detection](#8) |
| 9 | [Pattern 5: Data Lineage Tracking](#9) |
| 10 | [The Data Quality Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Is Data Quality? The Visual Model

---

```
DATA QUALITY DIMENSIONS (the 6 pillars):

  COMPLETENESS:  required fields are non-null and non-empty
  VALIDITY:      values conform to schema (type, range, pattern)
  CONSISTENCY:   same entity has same values across tables
  ACCURACY:      values reflect the real world (hard to automate)
  TIMELINESS:    data arrives within expected SLA window
  UNIQUENESS:    no duplicates on key columns

DATA QUALITY PIPELINE:

  RAW DATA
      │
      ▼
  [SCHEMA VALIDATION]   ← type check, null check, column existence
      │  fail → quarantine to DLQ / alert
      ▼
  [STATISTICAL PROFILING] ← null rate, distinct count, min/max/mean/stddev
      │  anomaly → alert, hold for review
      ▼
  [EXPECTATION RULES]   ← business rules (amount > 0, status in allowed set)
      │  fail → reject row, route to failed_records table
      ▼
  [ANOMALY DETECTION]   ← z-score, rolling baseline drift detection
      │  alert on 3σ deviation
      ▼
  CURATED DATA

GREAT EXPECTATIONS CONCEPTS:
  Expectation:       one assertion (expect_column_not_null)
  ExpectationSuite:  named collection of expectations
  Checkpoint:        validates a batch of data against a suite
  Data Docs:         HTML report of all validation results

LINEAGE GRAPH:
  raw_orders.csv → transform_orders (Spark) → orders_fact (Redshift)
  orders_fact → revenue_report (dbt) → dashboards
  If orders_fact is stale → all downstream is stale (graph traversal)
```


<a id='2'></a>
## 2. Core Concepts — Setup

In [ ]:
import random
import math
import re
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Callable
from collections import defaultdict, Counter
from enum import Enum

random.seed(42)

# simulate a dataset of sales records with intentional quality issues
def make_dirty_dataset(n=1000):
    rows = []
    statuses = ['OPEN', 'CLOSED', 'PENDING', None]  # None = null
    for i in range(n):
        row = {
            'order_id':   i if random.random() > 0.02 else None,         # 2% null
            'customer_id': random.randint(1, 500),
            'status':      random.choice(statuses),                       # some nulls
            'amount':      round(random.uniform(-50, 5000), 2),           # some negatives
            'email':       f'user{i}@example.com' if random.random() > 0.05 else 'bad-email', # 5% invalid
            'order_date':  f'2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}' if random.random() > 0.03 else 'not-a-date',
            'region':      random.choice(['NORTH','SOUTH','EAST','WEST','UNKNOWN']),
        }
        rows.append(row)
    # inject duplicates
    for _ in range(20):
        rows.append(rows[random.randint(0, 99)])  # duplicate first 100
    return rows

DATASET = make_dirty_dataset(1000)
print(f"Dataset: {len(DATASET)} rows, {len(DATASET[0])} columns")
print(f"Sample row: {DATASET[0]}")
print("Setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
DATA QUALITY OPERATIONS
────────────────────────────────────────────────────────────────────────────
OPERATION                           WHAT IT DOES
────────────────────────────────────────────────────────────────────────────
validate_schema(df, schema)         check types, nullability, allowed values
profile(df)                         compute null%, distinct_count, min/max/mean
expect_column_not_null(col)         assert 0 nulls in column
expect_values_in_set(col, vals)     assert all values are in allowed set
expect_column_between(col, lo, hi)  assert all values in numeric range
expect_unique(col)                  assert no duplicate values
expect_regex(col, pattern)          assert values match regex
anomaly_zscore(series, threshold)   flag rows > threshold stddevs from mean
lineage.add_edge(src, dst, op)      record data flow src → dst via op
lineage.upstream(node)              all ancestors of node
lineage.downstream(node)            all descendants of node
────────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Run quality checks on samples only for critical data
❌  Block production for soft warnings (separate critical vs warning expectations)
❌  Log failures without routing to a failed_records table
❌  Check only at load time — validate at every pipeline stage
❌  Ignore schema drift — new columns from upstream can silently break pipelines
❌  Alert on every anomaly without tuning threshold — alert fatigue kills response
```


In [ ]:
# Core API demo: quick quality summary of the dirty dataset

def quick_profile(rows, columns):
    for col in columns:
        values     = [r[col] for r in rows]
        null_count = sum(1 for v in values if v is None)
        distinct   = len(set(v for v in values if v is not None))
        null_pct   = 100 * null_count / len(values)
        numeric    = [v for v in values if isinstance(v, (int, float)) and v is not None]
        if numeric:
            col_min = min(numeric)
            col_max = max(numeric)
            col_avg = sum(numeric) / len(numeric)
            extra   = f"min={col_min:.0f} max={col_max:.0f} avg={col_avg:.1f}"
        else:
            top_val = Counter(v for v in values if v is not None).most_common(1)
            extra   = f"top={top_val[0][0] if top_val else 'N/A'}"
        status = '🔴' if null_pct > 5 else ('🟡' if null_pct > 1 else '🟢')
        print(f"  {status} {col:15s}: null={null_pct:5.1f}% distinct={distinct:5} {extra}")

print("=== Dataset Quality Overview ===")
quick_profile(DATASET, list(DATASET[0].keys()))

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
────────────────────────────────────────────────────────────────────────────
Wrong data types from upstream               Schema validation (hard fail)
Unexpected null spike                        Statistical profiling + alert
Business rule (amount > 0)                   Expectation rule
Row count dropped 40% vs yesterday           Anomaly detection (z-score)
Need to trace bad data to source             Lineage graph traversal
Duplicate records                            Uniqueness expectation + dedup
New column appeared from upstream            Schema drift detection
Pipeline runs but produces wrong results     End-to-end reconciliation
GDPR: track all data copies                  Column-level lineage
────────────────────────────────────────────────────────────────────────────
```


<a id='5'></a>
## 5. 🧩 Pattern 1: Schema Validation

---

```
PROBLEM:
  Upstream sends a CSV. You need to verify it matches the expected schema
  before processing — column types, required fields, allowed values.

APPROACH:
  Define a schema contract. Validate every incoming batch against it.
  Hard fail on type errors. Warn on missing optional columns.

SCHEMA VALIDATION LAYERS:
  1. Column existence: all required columns present?
  2. Type validation:  every value in column is expected type?
  3. Null check:       non-nullable columns have no nulls?
  4. Value set:        categorical columns only use allowed values?
  5. Pattern:          string columns match regex (email, date, phone)?

SCHEMA DRIFT:
  New column in upstream → downstream doesn't know it → data loss
  Missing column from upstream → downstream expects it → null explosion
  Type change (int → string) → downstream cast fails
  Detection: compare incoming schema fingerprint vs registered schema

SLOW MOTION: schema validation on order row
  row = {'order_id': '123', 'amount': -50, 'status': 'INVALID', 'email': 'bad'}
  check 1: order_id expected int → '123' is string → FAIL (type error)
  check 2: amount expected > 0 → -50 → FAIL (range error)
  check 3: status expected in {OPEN,CLOSED,PENDING} → INVALID → FAIL
  check 4: email must match regex → 'bad' → FAIL
  → row rejected, routed to failed_records with reason codes

KEY INSIGHT:
  Schema validation is the first gate — fail fast before expensive processing.
  Store rejected rows in a dead_letter_queue table with reason codes for auditing.

TIME / SPACE:
  Schema validation: O(C × N) — C columns × N rows, all fast type checks
  Space: O(F) — F = failed rows (route to DLQ, don't discard)
```


In [ ]:
# Pattern 1: Schema validation

# Slow motion on first 5 rows: check each validation rule
# col='order_id': check not-null → rule passes/fails → collect reason
# col='amount':   check numeric and > 0 → negatives fail
# col='status':   check in allowed set → 'UNKNOWN' fails
# col='email':    check regex pattern → 'bad-email' fails

@dataclass
class ColumnSchema:
    name:      str
    dtype:     type          # expected Python type
    nullable:  bool = True
    allowed:   Optional[set] = None  # allowed value set
    min_val:   Optional[float] = None
    max_val:   Optional[float] = None
    regex:     Optional[str] = None

@dataclass
class ValidationResult:
    row_index: int
    row:       dict
    errors:    List[str]

class SchemaValidator:
    """
    Data Quality Pattern 1 — Schema validation.
    Approach: Apply layered rules per column; collect all failures per row.
    Time:  O(C × N) — C columns × N rows
    Space: O(F) — F failed rows in DLQ
    """
    def __init__(self, schema: List[ColumnSchema]):
        self.schema = {s.name: s for s in schema}
        self.passed: List[dict] = []
        self.failed: List[ValidationResult] = []

    def validate_row(self, idx, row) -> List[str]:
        errors = []
        for col_name, rule in self.schema.items():
            val = row.get(col_name)
            # null check
            if val is None:
                if not rule.nullable:
                    errors.append(f"{col_name}: required but null")
                continue
            # type check
            if not isinstance(val, rule.dtype):
                try:
                    rule.dtype(val)  # attempt coercion
                except (ValueError, TypeError):
                    errors.append(f"{col_name}: expected {rule.dtype.__name__}, got {type(val).__name__}")
                    continue
            # range check
            if rule.min_val is not None and isinstance(val, (int, float)) and val < rule.min_val:
                errors.append(f"{col_name}: {val} < min {rule.min_val}")
            if rule.max_val is not None and isinstance(val, (int, float)) and val > rule.max_val:
                errors.append(f"{col_name}: {val} > max {rule.max_val}")
            # allowed set check
            if rule.allowed and val not in rule.allowed:
                errors.append(f"{col_name}: '{val}' not in allowed set")
            # regex check
            if rule.regex and isinstance(val, str) and not re.match(rule.regex, val):
                errors.append(f"{col_name}: '{val}' fails regex {rule.regex}")
        return errors

    def validate_batch(self, rows):
        self.passed = []
        self.failed = []
        for idx, row in enumerate(rows):
            errors = self.validate_row(idx, row)
            if errors:
                self.failed.append(ValidationResult(idx, row, errors))
            else:
                self.passed.append(row)
        return self

    def report(self):
        total = len(self.passed) + len(self.failed)
        print(f"  Validation: {len(self.passed)}/{total} passed  {len(self.failed)} failed ({100*len(self.failed)/total:.1f}%)")

ORDER_SCHEMA = [
    ColumnSchema('order_id',   int,   nullable=False),
    ColumnSchema('customer_id',int,   nullable=False, min_val=1),
    ColumnSchema('status',     str,   nullable=False, allowed={'OPEN','CLOSED','PENDING'}),
    ColumnSchema('amount',     float, nullable=False, min_val=0.0, max_val=99999.0),
    ColumnSchema('email',      str,   nullable=True,  regex=r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$'),
    ColumnSchema('order_date', str,   nullable=True,  regex=r'^\d{4}-\d{2}-\d{2}$'),
]

validator = SchemaValidator(ORDER_SCHEMA)
validator.validate_batch(DATASET)
validator.report()

print()
print("=== Sample Validation Failures ===")
for vr in validator.failed[:5]:
    print(f"  Row {vr.row_index}: {vr.errors}")

print()
# schema drift detection
print("=== Schema Drift Detection ===")
expected_cols = set(s.name for s in ORDER_SCHEMA)
actual_cols   = set(DATASET[0].keys())
new_cols      = actual_cols - expected_cols
missing_cols  = expected_cols - actual_cols
print(f"  Expected columns: {sorted(expected_cols)}")
print(f"  Actual columns:   {sorted(actual_cols)}")
print(f"  New (unexpected): {new_cols if new_cols else 'none'}")
print(f"  Missing:          {missing_cols if missing_cols else 'none'}")

print("\nSchema validation pattern complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Statistical Profiling

---

```
PROBLEM:
  Before loading data, you need a comprehensive quality report:
  null rates, value distributions, outliers, duplicate rates.

APPROACH:
  Statistical profiling = automated EDA (exploratory data analysis).
  Run on every batch. Compare to baseline. Alert on deviation.

METRICS TO COMPUTE:
  Numeric columns:
    null_rate, mean, stddev, min, p25, p50, p75, p95, p99, max
  Categorical columns:
    null_rate, distinct_count, top_N_values, cardinality_ratio
  All columns:
    completeness, uniqueness (distinct / total)

BASELINE DRIFT DETECTION:
  Store profiles from last N runs.
  Alert if current metric deviates > threshold from baseline.
  Example: null_rate: baseline=2%, current=15% → alert

SLOW MOTION: profile 'amount' column
  values = [10, 50, 99, -5, 200, null, 3000, ...]
  null_rate = null_count / total
  sorted → percentiles via index: p50 = sorted[n//2]
  mean = sum(non-null) / count(non-null)
  stddev = sqrt(mean of squared deviations)
  outliers: values > mean + 3×stddev

KEY INSIGHT:
  Profile every batch. A sudden jump in null_rate or mean signals upstream change.
  Automated profiling finds issues before users report them.

TIME / SPACE:
  Profile:    O(N log N) — sort for percentiles + single pass for moments
  Comparison: O(C × K) — C columns × K baseline metrics
  Storage:    O(C × N_runs) for baseline history (small — just statistics)
```


In [ ]:
# Pattern 2: Statistical profiling

# Slow motion on 'amount' column:
# step 1: separate null and non-null values
# step 2: compute mean, stddev via single pass
# step 3: sort non-null values → percentiles via index
# step 4: flag outliers (value > mean + 3σ)

@dataclass
class ColumnProfile:
    name:         str
    dtype:        str  # 'numeric' or 'categorical'
    row_count:    int
    null_count:   int
    null_rate:    float
    distinct:     int
    # numeric only
    mean:         Optional[float] = None
    stddev:       Optional[float] = None
    p0:           Optional[float] = None
    p25:          Optional[float] = None
    p50:          Optional[float] = None
    p75:          Optional[float] = None
    p99:          Optional[float] = None
    p100:         Optional[float] = None
    # categorical only
    top_values:   Optional[List] = None

class DataProfiler:
    """
    Data Quality Pattern 2 — Statistical profiling.
    Approach: Compute descriptive stats per column; compare to stored baseline.
    Time:  O(N log N) per numeric column (sort for percentiles)
    Space: O(C) — one ColumnProfile per column
    """
    def profile_column(self, name, values) -> ColumnProfile:
        total      = len(values)
        null_vals  = [v for v in values if v is None]
        nonnull    = [v for v in values if v is not None]
        null_rate  = len(null_vals) / total if total else 0
        distinct   = len(set(nonnull))

        # determine if numeric
        numeric_vals = [v for v in nonnull if isinstance(v, (int, float))]
        if len(numeric_vals) > len(nonnull) * 0.8:  # mostly numeric
            sorted_v = sorted(numeric_vals)
            n = len(sorted_v)
            mean   = sum(sorted_v) / n if n else 0
            var    = sum((x - mean) ** 2 for x in sorted_v) / n if n else 0
            stddev = math.sqrt(var)
            def pct(p):
                return sorted_v[max(0, int(n * p) - 1)] if n else None
            return ColumnProfile(name=name, dtype='numeric', row_count=total,
                null_count=len(null_vals), null_rate=null_rate, distinct=distinct,
                mean=mean, stddev=stddev,
                p0=pct(0.01), p25=pct(0.25), p50=pct(0.50),
                p75=pct(0.75), p99=pct(0.99), p100=sorted_v[-1] if sorted_v else None)
        else:
            top = Counter(nonnull).most_common(5)
            return ColumnProfile(name=name, dtype='categorical', row_count=total,
                null_count=len(null_vals), null_rate=null_rate, distinct=distinct,
                top_values=top)

    def profile(self, rows) -> Dict[str, ColumnProfile]:
        cols = rows[0].keys() if rows else []
        return {col: self.profile_column(col, [r[col] for r in rows]) for col in cols}

    def compare(self, current, baseline, alert_threshold=0.5):
        # drift = (current - baseline) / baseline; alert if > threshold
        alerts = []
        for col in current:
            if col not in baseline:
                continue
            c, b = current[col], baseline[col]
            if b.null_rate > 0:
                drift = abs(c.null_rate - b.null_rate) / b.null_rate
                if drift > alert_threshold:
                    alerts.append(f"{col}: null_rate drifted {b.null_rate:.1%} → {c.null_rate:.1%} ({drift:.0%} change)")
            if b.mean and c.mean:
                mean_drift = abs(c.mean - b.mean) / (abs(b.mean) + 1e-9)
                if mean_drift > alert_threshold:
                    alerts.append(f"{col}: mean drifted {b.mean:.1f} → {c.mean:.1f} ({mean_drift:.0%} change)")
        return alerts

profiler = DataProfiler()
profiles = profiler.profile(DATASET)

print("=== Statistical Profiles ===")
for col, p in profiles.items():
    if p.dtype == 'numeric':
        print(f"  {col:15s} [numeric]: null={p.null_rate:.1%} mean={p.mean:.1f} std={p.stddev:.1f} p50={p.p50:.1f} p99={p.p99:.1f}")
    else:
        top_str = ', '.join(f"{v}({c})" for v,c in (p.top_values or [])[:3])
        print(f"  {col:15s} [categ]:   null={p.null_rate:.1%} distinct={p.distinct} top=[{top_str}]")

print()
# simulate baseline comparison
print("=== Drift Detection vs Yesterday's Baseline ===")
# manually tweak some profiles to simulate drift
baseline_profiles = dict(profiles)
import copy
today_profiles = copy.deepcopy(profiles)
today_profiles['amount'].null_rate = 0.18  # spike from 2% to 18%
today_profiles['amount'].mean = 1200.0     # mean jumped (price increase)

alerts = profiler.compare(today_profiles, baseline_profiles, alert_threshold=0.3)
if alerts:
    print("  ⚠️  DRIFT ALERTS:")
    for alert in alerts:
        print(f"    - {alert}")
else:
    print("  All metrics within threshold")

print("\nStatistical profiling pattern complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Expectation-Based Validation (Great Expectations style)

---

```
PROBLEM:
  Business says: 'amount must always be positive', 'status must be in {OPEN, CLOSED, PENDING}',
  'email must match pattern'. Encode these as testable, versioned rules.

APPROACH:
  Expectation = one testable assertion about a column or dataset.
  ExpectationSuite = named, versioned collection of expectations.
  Checkpoint = run suite against a batch → produce validation result.

EXPECTATION TYPES:
  Completeness: expect_column_values_to_not_be_null
  Range:        expect_column_values_to_be_between(min, max)
  Set:          expect_column_values_to_be_in_set(value_set)
  Regex:        expect_column_values_to_match_regex(pattern)
  Uniqueness:   expect_column_values_to_be_unique
  Row count:    expect_table_row_count_to_be_between(min, max)
  Type:         expect_column_values_to_be_of_type(type_str)

SEVERITY LEVELS:
  CRITICAL (fail pipeline):  wrong types, unexpected nulls in PKs
  WARNING (alert only):      null rate spike, unexpected categories
  INFO (log only):           count drift, new values in set

SLOW MOTION: checkpoint run
  batch = 1000 rows
  run expectation: amount > 0
    → count values where amount <= 0 → 15 failures (1.5%)
    → unexpected_count = 15, unexpected_percent = 1.5%
    → success = True if unexpected_percent <= 0 else False
  aggregate all expectations → ValidationResult(success=True/False)

KEY INSIGHT:
  Expectations as code = data contracts you can version-control.
  Bad data that passes expectations is a contract problem — update the expectation.
  Bad data that fails expectations is a pipeline problem — fix the source.

TIME / SPACE:
  Per expectation: O(N) — one pass over column values
  Suite run: O(E × N) — E expectations × N rows
  Failed rows: O(F) — store for DLQ analysis
```


In [ ]:
# Pattern 3: Expectation-based validation (Great Expectations style)

# Slow motion on a batch of orders:
# expect_not_null('order_id') → count nulls → unexpected_count, pct
# expect_between('amount', 0, 99999) → count out-of-range → pass/fail
# expect_in_set('status', {...}) → count unlisted values → pass/fail
# aggregate all expectations → suite result (success if all critical pass)

from enum import Enum

class Severity(Enum):
    CRITICAL = 'CRITICAL'
    WARNING  = 'WARNING'
    INFO     = 'INFO'

@dataclass
class ExpectationResult:
    name:               str
    success:            bool
    unexpected_count:   int
    unexpected_pct:     float
    severity:           Severity
    sample_failures:    List[Any] = field(default_factory=list)

class ExpectationSuite:
    """
    Data Quality Pattern 3 — Expectation-based validation.
    Approach: Declare assertions; evaluate against batch; collect pass/fail per row.
    Time:  O(E × N) — E expectations × N rows
    Space: O(F) — failed rows cached for DLQ
    """
    def __init__(self, name):
        self.name = name
        self.expectations: List[Dict] = []

    def expect_not_null(self, col, severity=Severity.CRITICAL):
        self.expectations.append({'type': 'not_null', 'col': col, 'severity': severity})

    def expect_between(self, col, lo, hi, severity=Severity.CRITICAL):
        self.expectations.append({'type': 'between', 'col': col, 'lo': lo, 'hi': hi, 'severity': severity})

    def expect_in_set(self, col, value_set, severity=Severity.WARNING):
        self.expectations.append({'type': 'in_set', 'col': col, 'set': set(value_set), 'severity': severity})

    def expect_regex(self, col, pattern, severity=Severity.WARNING):
        self.expectations.append({'type': 'regex', 'col': col, 'pattern': pattern, 'severity': severity})

    def expect_unique(self, col, severity=Severity.CRITICAL):
        self.expectations.append({'type': 'unique', 'col': col, 'severity': severity})

    def run(self, rows) -> List[ExpectationResult]:
        results = []
        for exp in self.expectations:
            col     = exp['col']
            values  = [r[col] for r in rows]
            failures = []
            if exp['type'] == 'not_null':
                failures = [v for v in values if v is None]
            elif exp['type'] == 'between':
                failures = [v for v in values if v is not None and not (exp['lo'] <= v <= exp['hi'])]
            elif exp['type'] == 'in_set':
                failures = [v for v in values if v is not None and v not in exp['set']]
            elif exp['type'] == 'regex':
                failures = [v for v in values if v is not None and not re.match(exp['pattern'], str(v))]
            elif exp['type'] == 'unique':
                seen = Counter(v for v in values if v is not None)
                failures = [v for v, c in seen.items() if c > 1]
            pct = 100 * len(failures) / max(len(values), 1)
            results.append(ExpectationResult(
                name=f"{exp['type']}({col})",
                success=(len(failures) == 0),
                unexpected_count=len(failures),
                unexpected_pct=pct,
                severity=exp['severity'],
                sample_failures=failures[:3]
            ))
        return results

suite = ExpectationSuite('orders_v1')
suite.expect_not_null('order_id',    Severity.CRITICAL)
suite.expect_not_null('status',      Severity.CRITICAL)
suite.expect_between('amount',   0, 99999, Severity.CRITICAL)
suite.expect_in_set('status', {'OPEN','CLOSED','PENDING'}, Severity.WARNING)
suite.expect_regex('email', r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$', Severity.WARNING)
suite.expect_unique('order_id',      Severity.CRITICAL)

results = suite.run(DATASET)

print(f"=== Expectation Suite: '{suite.name}' ({len(DATASET)} rows) ===")
all_critical_passed = True
for r in results:
    icon  = '✅' if r.success else ('🔴' if r.severity == Severity.CRITICAL else '🟡')
    print(f"  {icon} {r.name:35s}: {'PASS' if r.success else 'FAIL':4s} {r.unexpected_count:4d} failures ({r.unexpected_pct:.1f}%) [{r.severity.value}]")
    if not r.success and r.sample_failures:
        print(f"    sample failures: {r.sample_failures}")
    if not r.success and r.severity == Severity.CRITICAL:
        all_critical_passed = False

print()
print(f"  Overall: {'✅ PASS — pipeline proceeds' if all_critical_passed else '🔴 FAIL — pipeline BLOCKED (critical expectations failed)'}")

print("\nExpectation-based validation pattern complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Anomaly Detection

---

```
PROBLEM:
  Row count was 1M yesterday. Today it's 600k. No schema errors.
  How do you detect and alert on this kind of unexpected drop?

APPROACH:
  Statistical anomaly detection: compare today's metrics to rolling baseline.
  Two methods:
    1. Z-score:       (value - mean) / stddev > threshold → anomaly
    2. Percentage:    abs(today - yesterday) / yesterday > threshold

Z-SCORE DETECTION:
  window = last 14 days of row counts
  baseline_mean   = mean(window)
  baseline_stddev = stddev(window)
  today_zscore = (today_count - baseline_mean) / baseline_stddev
  alert if z < -3 (significant drop) or z > 3 (unexpected spike)

SLOW MOTION: z-score on row count
  window = [1M, 1.01M, 0.98M, 1.02M, 0.99M, 1.01M, 1M, 0.97M, ...]  (14 days)
  mean   = 1,000,000
  stddev = 15,000  (tight distribution)
  today  = 600,000
  z = (600k - 1M) / 15k = -26.7 → far below -3 threshold → CRITICAL ALERT

METRICS TO MONITOR:
  row_count:      daily count per table/partition
  null_rate:      per critical column
  mean/stddev:    for key numeric columns (revenue, quantity)
  distinct_count: for key categorical columns (customer_id cardinality)
  load_duration:  ETL job duration (slow → resource issue)

KEY INSIGHT:
  Absolute thresholds (count < 900k) break over time as data grows.
  Z-score automatically adapts to natural growth trends.
  Use percentage thresholds for business metrics, z-score for technical ones.

TIME / SPACE:
  Z-score compute: O(W) per metric — W = window size (14 days)
  Alert check:     O(M) — M metrics monitored
```


In [ ]:
# Pattern 4: Anomaly detection — Z-score and percentage threshold

# Slow motion: row_count anomaly via z-score
# step 1: collect 14-day window of daily row counts
# step 2: compute mean and stddev of window
# step 3: compute z-score for today's count
# step 4: flag if z < -3 or z > 3

@dataclass
class MetricPoint:
    date:   str
    metric: str
    value:  float

class AnomalyDetector:
    """
    Data Quality Pattern 4 — Anomaly detection.
    Approach: Rolling z-score and percentage deviation from window baseline.
    Time:  O(W) per metric check — W = window size
    Space: O(W × M) — W days × M metrics stored in baseline history
    """
    def __init__(self, window_days=14, zscore_threshold=3.0, pct_threshold=0.30):
        self.window        = window_days
        self.zscore_thresh = zscore_threshold
        self.pct_thresh    = pct_threshold
        self.history: Dict[str, List[float]] = defaultdict(list)

    def add_point(self, metric, value):
        self.history[metric].append(value)
        if len(self.history[metric]) > self.window:
            self.history[metric].pop(0)  # slide the window

    def check(self, metric, today_value) -> Dict:
        window = self.history[metric]
        if len(window) < 3:
            return {'anomaly': False, 'reason': 'insufficient history'}
        mean   = sum(window) / len(window)
        var    = sum((x - mean)**2 for x in window) / len(window)
        stddev = math.sqrt(var) if var > 0 else 1
        zscore = (today_value - mean) / stddev
        prev   = window[-1]
        pct_change = abs(today_value - prev) / max(abs(prev), 1)
        anomaly    = abs(zscore) > self.zscore_thresh or pct_change > self.pct_thresh
        return {
            'anomaly':    anomaly,
            'zscore':     round(zscore, 2),
            'pct_change': round(pct_change * 100, 1),
            'baseline_mean': round(mean, 1),
            'today':      today_value,
            'reason':     f"z={zscore:.1f}, pct={pct_change*100:.0f}%" if anomaly else 'within bounds'
        }

detector = AnomalyDetector(window_days=14, zscore_threshold=3.0, pct_threshold=0.30)

# simulate 14 days of normal data
print("=== Building 14-day Baseline ===")
daily_counts = [random.randint(950000, 1050000) for _ in range(14)]
for day, count in enumerate(daily_counts):
    detector.add_point('row_count_orders', count)
print(f"  14-day baseline: mean≈{sum(daily_counts)/len(daily_counts):.0f}, range=[{min(daily_counts):,}, {max(daily_counts):,}]")

print()
print("=== Anomaly Checks ===")

# normal day
result = detector.check('row_count_orders', 1_010_000)
icon = '🔴 ANOMALY' if result['anomaly'] else '✅ Normal'
print(f"  count=1,010,000 (normal day): {icon} | {result['reason']}")

# 40% drop
result2 = detector.check('row_count_orders', 600_000)
icon2 = '🔴 ANOMALY' if result2['anomaly'] else '✅ Normal'
print(f"  count=600,000 (40% drop):     {icon2} | z={result2['zscore']} pct={result2['pct_change']}% baseline={result2['baseline_mean']:.0f}")

# spike
result3 = detector.check('row_count_orders', 2_000_000)
icon3 = '🔴 ANOMALY' if result3['anomaly'] else '✅ Normal'
print(f"  count=2,000,000 (2× spike):   {icon3} | z={result3['zscore']} pct={result3['pct_change']}%")

print()
print("=== Multi-Metric Monitoring ===")
metrics_to_monitor = [
    ('row_count_orders',  1_010_000, True,  'row count normal'),
    ('row_count_returns',    50_000, True,  'returns normal'),
    ('null_rate_amount',        0.03, False, 'null rate — %'),
    ('avg_order_value',        150.0, False, 'avg order value'),
]
for metric, today_val, is_count, desc in metrics_to_monitor:
    # add some history
    baseline_val = today_val * random.uniform(0.95, 1.05)
    for _ in range(14):
        detector.add_point(metric, baseline_val * random.uniform(0.97, 1.03))
    result = detector.check(metric, today_val)
    icon = '🔴' if result['anomaly'] else '🟢'
    print(f"  {icon} {metric:25s}: {result['reason']}")

print("\nAnomaly detection pattern complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Data Lineage Tracking

---

```
PROBLEM:
  A BI dashboard shows wrong revenue numbers.
  You need to trace which upstream table caused it,
  and determine which other dashboards are also affected.

APPROACH:
  Lineage graph: directed graph where nodes are datasets/tables,
  edges are transformations (ETL jobs, SQL models).
  Upstream traversal → find root cause.
  Downstream traversal → impact analysis.

LINEAGE TYPES:
  Table-level:   orders_raw → orders_cleaned → revenue_summary → dashboard
  Column-level:  orders.amount → revenue_summary.total_revenue → report.revenue
  Job-level:     which ETL job produced which table

UPSTREAM TRAVERSAL (root cause):
  Problem: dashboard shows wrong revenue
  Upstream(dashboard) → revenue_summary → orders_cleaned → orders_raw
  Investigate in reverse order: orders_raw was the bad data

DOWNSTREAM TRAVERSAL (impact):
  orders_raw is stale:
  Downstream(orders_raw) → orders_cleaned → [revenue_summary, shipping_report]
                         → revenue_summary → [dashboard_A, dashboard_B]
  Impact: 4 nodes affected

KEY INSIGHT:
  Without lineage, a data quality incident takes hours to trace.
  With lineage, the same trace takes seconds — just traverse the graph.
  Lineage also powers GDPR compliance: trace all copies of a user's PII.

TIME / SPACE:
  Graph traversal: O(V + E) — BFS/DFS over nodes + edges
  Storage: O(V + E) — one record per table + one per transformation edge
```


In [ ]:
# Pattern 5: Data lineage tracking

# Slow motion: impact analysis when orders_raw has bad data
# step 1: add edge orders_raw → orders_cleaned (Spark job)
# step 2: add edge orders_cleaned → revenue_summary (dbt model)
# step 3: add edge revenue_summary → dashboard_A (BI tool)
# step 4: downstream(orders_raw) → BFS → find all affected nodes

class LineageGraph:
    """
    Data Quality Pattern 5 — Data lineage tracking.
    Approach: Directed graph of dataset nodes and transformation edges.
    Time:  O(V + E) for traversal (BFS/DFS)
    Space: O(V + E) for graph storage
    """
    def __init__(self):
        self.nodes: Dict[str, Dict] = {}   # node_id → metadata
        self.edges: List[Dict] = []         # src → dst via operation
        self.adj_out: Dict[str, List[str]] = defaultdict(list)  # forward edges
        self.adj_in:  Dict[str, List[str]] = defaultdict(list)  # backward edges

    def add_node(self, node_id, node_type='table', owner=None, sla_minutes=None):
        self.nodes[node_id] = {'type': node_type, 'owner': owner, 'sla': sla_minutes}

    def add_edge(self, src, dst, operation='ETL', job_id=None):
        self.edges.append({'src': src, 'dst': dst, 'op': operation, 'job': job_id})
        self.adj_out[src].append(dst)
        self.adj_in[dst].append(src)

    def upstream(self, node_id) -> List[str]:
        # BFS traversal backward (follow in-edges)
        visited = set()
        queue   = deque([node_id])
        result  = []
        while queue:
            n = queue.popleft()
            for parent in self.adj_in.get(n, []):
                if parent not in visited:
                    visited.add(parent)
                    result.append(parent)
                    queue.append(parent)
        return result

    def downstream(self, node_id) -> List[str]:
        # BFS traversal forward (follow out-edges)
        visited = set()
        queue   = deque([node_id])
        result  = []
        while queue:
            n = queue.popleft()
            for child in self.adj_out.get(n, []):
                if child not in visited:
                    visited.add(child)
                    result.append(child)
                    queue.append(child)
        return result

# build a sample lineage graph
g = LineageGraph()
# source tables
for node in ['orders_raw', 'customers_raw', 'products_raw']:
    g.add_node(node, 'raw_source', owner='data-ingestion-team')
# curated tables
for node in ['orders_cleaned', 'customers_cleaned', 'dim_products']:
    g.add_node(node, 'curated', owner='data-platform-team')
# analytical tables
for node in ['revenue_summary', 'customer_360', 'inventory_report']:
    g.add_node(node, 'analytical', owner='analytics-team')
# serving layer
for node in ['dashboard_revenue', 'dashboard_ops', 'weekly_report']:
    g.add_node(node, 'dashboard', owner='bi-team')

# transformation edges
g.add_edge('orders_raw',     'orders_cleaned',  'Spark ETL',  'job_clean_orders')
g.add_edge('customers_raw',  'customers_cleaned','Spark ETL',  'job_clean_customers')
g.add_edge('products_raw',   'dim_products',     'dbt model',  'dbt_dim_products')
g.add_edge('orders_cleaned', 'revenue_summary',  'dbt model',  'dbt_revenue')
g.add_edge('orders_cleaned', 'customer_360',     'dbt model',  'dbt_c360')
g.add_edge('customers_cleaned','customer_360',   'dbt model',  'dbt_c360')
g.add_edge('orders_cleaned', 'inventory_report', 'Spark',      'job_inventory')
g.add_edge('revenue_summary','dashboard_revenue','BI tool',    None)
g.add_edge('revenue_summary','weekly_report',    'BI export',  None)
g.add_edge('customer_360',   'dashboard_ops',    'BI tool',    None)

print("=== Lineage Graph ===")
print(f"  Nodes: {len(g.nodes)}  Edges: {len(g.edges)}")

print()
print("=== Root Cause Analysis: dashboard_revenue shows wrong data ===")
upstream = g.upstream('dashboard_revenue')
print(f"  Upstream of dashboard_revenue: {upstream}")
print(f"  Investigate: check orders_raw first (furthest upstream)")

print()
print("=== Impact Analysis: orders_raw is stale ===")
affected = g.downstream('orders_raw')
print(f"  Downstream of orders_raw: {affected}")
print(f"  {len(affected)} nodes affected — notify owners: {set(g.nodes[n]['owner'] for n in affected if n in g.nodes)}")

print()
print("=== GDPR: trace all copies of customer data ===")
customer_copies = g.downstream('customers_raw')
print(f"  All datasets containing customer data: {['customers_raw'] + customer_copies}")
print(f"  (Each must be included in GDPR deletion request)")

print("\nData lineage pattern complete.")

<a id='10'></a>
## 10. The Data Quality Decision Map

---

```
PROBLEM                                    PATTERN         TOOL
────────────────────────────────────────────────────────────────────────────
Wrong types from upstream                  Schema validation  Custom / GE
Required field is null                     Schema validation  Custom / GE
Business rule (amount > 0)                 Expectation        GE / dbt test
Null rate spiked                           Statistical profil. GE / custom
Row count dropped 40%                      Anomaly detection  Z-score
Mean revenue jumped unexpectedly           Anomaly detection  Z-score
Trace bad data to source                   Lineage traversal  OpenLineage/dbt
Impact of table deprecation                Lineage downstream OpenLineage
GDPR: find all copies of PII               Lineage downstream Column lineage
New columns from upstream (schema drift)   Schema diff alert  Custom
────────────────────────────────────────────────────────────────────────────

TOOLING LANDSCAPE:
  Great Expectations:  expectation-based validation, Data Docs HTML reports
  dbt tests:           schema tests (not_null, unique, accepted_values, relationships)
  OpenLineage:         open standard for lineage events (Marquez backend)
  DataHub:             enterprise data catalog + lineage UI
  Monte Carlo:         commercial anomaly detection + lineage
  Soda:                SQL-based expectations, simpler than GE

DQ SEVERITY FRAMEWORK:
  CRITICAL: fail pipeline + page on-call + block dashboard refresh
    → null PK, wrong type, row count 0, schema missing columns
  WARNING:  alert Slack + quarantine to review table
    → null rate spike, unexpected category value, anomaly z > 3
  INFO:     log to DQ metrics dashboard
    → minor drift, new value in set, new column detected
```


<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for each pattern:

| Signal | Pattern |
|--------|----------|
| "Validate types/nulls" | Schema validation |
| "Understand data distribution" | Statistical profiling |
| "Encode business rules" | Expectation suite (GE / dbt) |
| "Row count dropped" | Anomaly detection (z-score) |
| "Trace bad data" | Lineage upstream traversal |
| "Impact of table change" | Lineage downstream traversal |

---

### Key metrics — memorize these:

```
null_rate:       % of values that are null per column
completeness:    1 - null_rate
uniqueness:      distinct_count / total_count
cardinality:     distinct_count (low = good for compression)
z-score:         (value - mean) / stddev
z > 3 or < -3:  statistically significant anomaly (0.3% expected naturally)
```

---

### Common templates:

```python
# TEMPLATE: dbt schema test
# schema.yml:
# columns:
#   - name: order_id
#     tests: [not_null, unique]
#   - name: status
#     tests:
#       - accepted_values:
#           values: ['OPEN', 'CLOSED', 'PENDING']

# TEMPLATE: Z-score anomaly check
mean   = sum(window) / len(window)
stddev = math.sqrt(sum((x-mean)**2 for x in window) / len(window))
zscore = (today_value - mean) / stddev
is_anomaly = abs(zscore) > 3

# TEMPLATE: Lineage impact query (OpenLineage)
# GET /api/v1/lineage?nodeId=table://db.orders_raw&depth=5
```

---

### Gotchas to not forget:

```
❌  Blocking pipeline on WARNING-level expectations — save that for CRITICAL
❌  Discarding failed rows without writing to DLQ — you lose the audit trail
❌  Running DQ checks only at load time — validate at every stage
❌  Absolute thresholds (count < 900k) — they break as data grows; use z-score
✅  Expectations as code → version-controlled, testable data contracts
✅  DQ checks in CI/CD → catch schema changes before production
✅  Lineage graph = fastest root cause path during incidents
✅  Profile every batch and store — compare current to 14-day rolling baseline
```


<a id='12'></a>
## 12. Summary Map

---

```
                  ✅ DATA QUALITY & OBSERVABILITY
                              │
         ┌────────────────────┼────────────────────┐
         │                    │                    │
   VALIDATE               MONITOR              TRACE
   (Patterns 1,3)         (Patterns 2,4)       (Pattern 5)
         │                    │                    │
  Schema validation    Profiling             Lineage graph
  Type/null/range      null_rate, mean       upstream (root cause)
  Expectation suite    distinct, p50/p99     downstream (impact)
  GE / dbt tests       Anomaly z-score       OpenLineage / DataHub
  DLQ for rejects      14-day baseline       GDPR tracing

DQ PIPELINE STAGE GATES:
  Ingest → [Schema Validation] → [Statistical Profile]
         → [Expectation Suite] → [Anomaly Check] → Curated

SEVERITY RESPONSE:
  CRITICAL → block pipeline + page on-call + halt dashboard
  WARNING  → Slack alert + quarantine + manual review
  INFO     → log metric → trending dashboard
```

---
*End of Data Quality & Observability Master Guide — Sean Edition*
